# 1. Interactive Computing in Jupyter: Widgets & Dynamic Plots

### What we will cover:
1. Basic `@interact` decorator with sliders and dropdowns.
2. Composing custom layouts with explicit widgets (`FloatSlider`, `IntSlider`, `Checkbox`, `VBox`) coupled with **`plotly`** for dynamic real-time filtering.
3. **Interactive GeoPlot on OpenTopoMap**: Loading station metadata and sensor measurements from the `data/` directory, aggregating features per station, and rendering the network over high-detail **OpenTopoMap** topographic contour and elevation relief tiles.
4. **Interactive Station Time-Series Explorer**: Selecting any station from the dataset to dynamically inspect its sensor readings.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from IPython.display import display
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set random seed for reproducibility
np.random.seed(42)
print("Packages imported successfully!")


Packages imported successfully!


## 1.1 `@interact`

The quickest way to add interactivity to any function in Jupyter is the `@interact` decorator.
It automatically inspects the argument types and generates suitable controls (integers become integer sliders, booleans become checkboxes, strings become textboxes, lists become dropdowns etc).

In [2]:
@interact(frequency=(0.5, 5.0, 0.25), amplitude=(0.5, 3.0, 0.5), wave_type=["sine", "cosine", "square"])
def plot_wave(frequency=1.5, amplitude=1.0, wave_type="sine"):
    t = np.linspace(0, 4 * np.pi, 500)
    
    if wave_type == "sine":
        y = amplitude * np.sin(frequency * t)
    elif wave_type == "cosine":
        y = amplitude * np.cos(frequency * t)
    else:  # square wave
        y = amplitude * np.sign(np.sin(frequency * t))
        
    plt.figure(figsize=(9, 3.5))
    plt.plot(t, y, color="tab:blue", lw=2)
    plt.title(f"{wave_type.capitalize()} Wave (Freq = {frequency} Hz, Amp = {amplitude})")
    plt.xlabel("Time (t)")
    plt.ylabel("Signal")
    plt.grid(True, alpha=0.3)
    plt.ylim(-3.5, 3.5)
    plt.show()

interactive(children=(FloatSlider(value=1.5, description='frequency', max=5.0, min=0.5, step=0.25), FloatSlide…

## 1.2 Interactive Sensor Data Exploration with Widgets

In real data science challenges, sensor readings frequently suffer from:
- **High-frequency measurement noise**
- **Physical sensor clipping / outlier spikes**
- **Sensor dropouts / missing values**

In [3]:
# Generate a synthetic sensor baseline: daily temperature cycle + seasonal drift + noise + spikes
n_points = 300
n_spikes = int(n_points*0.1)

time_index = pd.date_range("2026-01-01", periods=n_points, freq="h")

trend = np.linspace(15, 25, n_points)
daily_cycle = 8 * np.sin(2 * np.pi * np.arange(n_points) / 24)
noise = np.random.normal(0, 1.2, size=n_points)

raw_sensor_values = trend + daily_cycle + noise

# Inject sensor anomalies
spike_indices = np.random.randint(n_points-1, size=n_spikes)
raw_sensor_values[spike_indices] = np.mean(raw_sensor_values) + \
                                   0.5 + np.random.rand(n_spikes) * \
                                   np.random.choice(np.array([-1,1]) * 5*np.std(raw_sensor_values), size=n_spikes)

sensor_df = pd.DataFrame({"timestamp": time_index, "raw_temperature": raw_sensor_values})
print(f"Generated {len(sensor_df)} sensor records with {len(spike_indices)} injected spikes.")
sensor_df.head()

Generated 300 sensor records with 30 injected spikes.


,timestamp,raw_temperature
0,2026-01-01 00:00:00,15.596057
1,2026-01-01 01:00:00,16.938080
2,2026-01-01 02:00:00,19.844116
3,2026-01-01 03:00:00,22.584825
4,2026-01-01 04:00:00,21.780998


### Interactive `ipywidgets` with `@interact` and `plotly`

We pass explicit `ipywidgets` controls into `@interact` to configure ranges and descriptions cleanly without manual `widgets.Output()` or `.observe()` wiring. Here we use **`plotly`** for interactive exploration with unified hover tooltips, zoom, and pan.

Compare **Naive Physical Clipping** (fixed global min/max thresholds) with **Trend-based Clipping** (two linear bound functions $y_{\text{lower}}(t)$ and $y_{\text{upper}}(t)$) to observe how trend awareness catches localized sensor anomalies without distorting the diurnal cycle.

In [ ]:
# UI controls
rolling_toggle = widgets.Checkbox(
    value=True,
    description="Enable Rolling Smoothing",
    style={"description_width": "initial"}
)

rolling_window_slider = widgets.IntSlider(
    value=12, min=2, max=24, step=1,
    description="Roll Window (h):",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="320px")
)

min_clip_slider = widgets.FloatSlider(
    value=5.0, min=-30.0, max=20.0, step=1.0,
    description="Min Valid Temp (°C):",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="320px")
)

max_clip_slider = widgets.FloatSlider(
    value=35.0, min=20.0, max=70.0, step=1.0,
    description="Max Valid Temp (°C):",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="320px")
)

show_raw_checkbox = widgets.Checkbox(
    value=True,
    description="Show Raw Signal & Spikes",
    style={"description_width": "initial"}
)

trend_toggle = widgets.Checkbox(
    value=True,
    description="Enable Trend-based Clipping",
    style={"description_width": "initial"}
)

lower_start_slider = widgets.FloatSlider(
    value=5.0, min=-30.0, max=30.0, step=0.5,
    description="Lower Start (°C):",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="320px")
)

lower_end_slider = widgets.FloatSlider(
    value=15.0, min=-30.0, max=40.0, step=0.5,
    description="Lower End (°C):",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="320px")
)

upper_start_slider = widgets.FloatSlider(
    value=25.0, min=0.0, max=70.0, step=0.5,
    description="Upper Start (°C):",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="320px")
)

upper_end_slider = widgets.FloatSlider(
    value=35.0, min=0.0, max=70.0, step=0.5,
    description="Upper End (°C):",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="320px")
)

def update_plot(
    use_trend,
    min_clip,
    max_clip,
    lower_start,
    lower_end,
    upper_start,
    upper_end,
    use_rolling,
    rolling_window,
    show_raw,
):
    # needed for disabling controlls for trend based, based on the value a toggle swtich
    min_clip_slider.disabled = use_trend
    max_clip_slider.disabled = use_trend
    lower_start_slider.disabled = not use_trend
    lower_end_slider.disabled = not use_trend
    upper_start_slider.disabled = not use_trend
    upper_end_slider.disabled = not use_trend
    rolling_window_slider.disabled = not use_rolling

    raw = sensor_df["raw_temperature"]

    if use_trend:
        # Two linear bound functions: everything outside is clipped to the bounds
        lower_line = pd.Series(
            np.linspace(lower_start, lower_end, len(raw)),
            index=raw.index,
        )
        upper_line = pd.Series(
            np.linspace(upper_start, upper_end, len(raw)),
            index=raw.index,
        )
        lower_bound = np.minimum(lower_line, upper_line)
        upper_bound = np.maximum(lower_line, upper_line)

        outlier_mask = (raw < lower_bound) | (raw > upper_bound)
        clipped = raw.clip(lower=lower_bound, upper=upper_bound)
    else:
        outlier_mask = (raw < min_clip) | (raw > max_clip)
        clipped = raw.clip(lower=min_clip, upper=max_clip)
    
    # Apply rolling mean filter from pandas when enabled
    if use_rolling:
        output_signal = clipped.rolling(window=rolling_window, center=True).mean()
        signal_name = f"Smoothed (w={rolling_window}h)"
    else:
        output_signal = clipped
        signal_name = "Clipped Signal"
    
    fig = go.Figure()
    
    # 1. Raw sensor data trace & detected anomalies
    if show_raw:
        fig.add_trace(go.Scatter(
            x=sensor_df["timestamp"],
            y=sensor_df["raw_temperature"],
            mode="lines",
            name="Raw Sensor Data",
            line=dict(color="#9d9eb3", width=1.2)
        ))
        
        spikes = sensor_df[outlier_mask]
        fig.add_trace(go.Scatter(
            x=spikes["timestamp"],
            y=spikes["raw_temperature"],
            mode="markers",
            name=f"Detected Anomalies ({len(spikes)})",
            marker=dict(color="crimson", size=7, symbol="circle", line=dict(width=1))
        ))
    
    # 2. Clipped / smoothed signal trace
    fig.add_trace(go.Scatter(
        x=sensor_df["timestamp"],
        y=output_signal, 
        mode="lines",
        name=signal_name,
        line=dict(color="#1f77b4", width=2.5)
    ))
    
    # 3. Reference lines / linear bounds
    if use_trend:
        fig.add_trace(go.Scatter(
            x=sensor_df["timestamp"],
            y=upper_bound,
            mode="lines",
            name=f"Upper Bound ({upper_start}→{upper_end}°C)",
            line=dict(color="rgba(214, 39, 40, 0.7)", width=1.5, dash="dash")
        ))
        fig.add_trace(go.Scatter(
            x=sensor_df["timestamp"],
            y=lower_bound,
            mode="lines",
            name=f"Lower Bound ({lower_start}→{lower_end}°C)",
            line=dict(color="rgba(214, 39, 40, 0.7)", width=1.5, dash="dash")
        ))
    else:
        fig.add_hline(
            y=min_clip,
            line_dash="dash",
            line_color="darkred",
            annotation_text=f"Min Limit ({min_clip}°C)",
            annotation_position="bottom left"
        )
        fig.add_hline(
            y=max_clip,
            line_dash="dash",
            line_color="darkred",
            annotation_text=f"Max Limit ({max_clip}°C)",
            annotation_position="top left"
        )

    # Layout of Widget
    fig.update_layout(
        title="Sensor Cleaning & Smoothing Dashboard (Plotly)",
        xaxis_title="Timestamp",
        yaxis_title="Temperature (°C)",
        template="plotly_white",
        hovermode="x unified",
        height=450,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1, font=dict(size=10), itemwidth=30)
    )
    fig.show()


# make the update_plot function reactive by wrapping in interactive() and setting all inputs to sliders.
# => doesnt redraw the whole plot object, just fills it with new data.
interactive_widgets = interactive(
    update_plot,
    use_trend=trend_toggle,
    min_clip=min_clip_slider,
    max_clip=max_clip_slider,
    lower_start=lower_start_slider,
    lower_end=lower_end_slider,
    upper_start=upper_start_slider,
    upper_end=upper_end_slider,
    use_rolling=rolling_toggle,
    rolling_window=rolling_window_slider,
    show_raw=show_raw_checkbox,
)

ui_controls = widgets.VBox([
    widgets.HBox([min_clip_slider, max_clip_slider]),
    trend_toggle,
    widgets.HBox([lower_start_slider, lower_end_slider]),
    widgets.HBox([upper_start_slider, upper_end_slider]),
    widgets.HBox([rolling_toggle, rolling_window_slider]),
    show_raw_checkbox,
])

# we arrange & display all sliders manually and just use the plot from interactive_widgets (ie the first element, the plotly object)
display(ui_controls, interactive_widgets.children[-1])


Output()

## 1.3 Interactive GeoPlot: Weather Station Network on OpenTopoMap

The 38 stations in our dataset are in a localized region in Styria, Austria (~15 km across), the standard geoplot map s only a flat, single-color polygon. To get geographic context, we render the stations over **OpenTopoMap** aka a "normal" map.


In [5]:
meta_path = Path("data/meta_data.csv")
data_dir = Path("data/original_provided_to_students")

meta_df = pd.read_csv(meta_path)

station_stats = []
for station_id in meta_df["Station"]:
    fpath = data_dir / f"station_{station_id}.csv"
    df = pd.read_csv(fpath)

    station_stats.append(
        {"Station":            int(station_id),
        "station_name":        f"Station {station_id}",
        "mean_temperature":    round(float(df["Air_temperature_degC"].mean()), 2),
        "mean_humidity":       round(float(df["Relative_humidity"].mean()), 2),
        "total_precipitation": round(float(df["Precipitation_mm"].sum()), 1),
        "mean_precipitation":  round(float(df["Precipitation_mm"].mean()), 3),
        "records_count":       len(df)}
    )

# Load data and merge on station (SQL inner join)
stats_df = pd.DataFrame(station_stats)
df_stations = meta_df.merge(stats_df, on="Station")
df_stations.head()


,Station,Lat,Long,Altitude,station_name,mean_temperature,mean_humidity,total_precipitation,mean_precipitation,records_count
0,1,46.9941,15.7611,328,Station 1,19.09,82.14,995.1,0.114,8782
1,2,46.9939,15.7790,312,Station 2,27.74,81.12,946.4,0.108,8782
2,3,46.9977,15.7923,393,Station 3,46.43,78.38,926.2,0.106,8782
3,4,46.9945,15.8135,398,Station 4,18.22,77.90,920.9,0.105,8782
4,5,46.9949,15.8361,326,Station 5,38.89,82.56,939.0,0.108,8782


### 1.3.1 Station Distribution on OpenTopoMap (Topographic Relief)

Below, each station is plotted over **OpenTopoMap** raster tiles. Dropdown for temperature, humidity & precipitation.


In [6]:
metric_dropdown = widgets.Dropdown(
    options=[
        ("Mean Temperature (°C)", "mean_temperature"),
        ("Altitude (m)", "Altitude"),
        ("Mean Relative Humidity (%)", "mean_humidity"),
        ("Total Precipitation (mm)", "total_precipitation"),
    ],
    value="mean_temperature",
    description="Display Metric:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="340px"),
)

@interact(metric=metric_dropdown)
def update_opentopo_map(metric):
    colorscale = "Plasma"
    
    fig_map = px.scatter_map(
        data_frame=df_stations,
        lat="Lat",
        lon="Long",
        color=metric,
        size="Altitude",
        opacity=0.7,
        color_continuous_scale=colorscale,
        hover_name="station_name",
        hover_data={
            "Station": True,
            "Altitude": ":.0f m",
            "mean_temperature": ":.2f °C",
            "mean_humidity": ":.1f %",
            "total_precipitation": ":.1f mm",
            "Lat": ":.4f",
            "Long": ":.4f",
        },
        zoom=11,
        center=dict(lat=float(df_stations["Lat"].mean()), lon=float(df_stations["Long"].mean())),
        title=f"Station Network (Scaled & Colored by {metric_dropdown.label})",
    )
    
    fig_map.update_layout(
        map=dict(
            style="white-bg",
            layers=[{
                "below": "traces",
                "sourcetype": "raster",
                "source": ["https://tile.opentopomap.org/{z}/{x}/{y}.png"],
            }],
        ),
        height=540,
        margin=dict(l=10, r=10, t=50, b=10),
    )
    fig_map.show()


interactive(children=(Dropdown(description='Display Metric:', layout=Layout(width='340px'), options=(('Mean Te…

## 1.4 Interactive Station Time-Series Explorer

In [7]:
station_selector = widgets.Dropdown(
    options=[(f"Station {s}", s) for s in sorted(df_stations["Station"])],
    value=1,
    description="Select Station:",
    layout=widgets.Layout(width="260px"),
)

channel_selector = widgets.Dropdown(
    options=[
        ("Air Temperature (°C)", "Air_temperature_degC"),
        ("Relative Humidity (%)", "Relative_humidity"),
        ("Precipitation (mm)", "Precipitation_mm"),
    ],
    value="Air_temperature_degC",
    description="Sensor Channel:",
    layout=widgets.Layout(width="280px"),
)

@interact(st_id=station_selector, var_col=channel_selector)
def update_station_ts(st_id, var_col):
    col_color = {
        "Air_temperature_degC": "#e6550d",
        "Relative_humidity": "#31a354",
        "Precipitation_mm": "#3182bd",
    }.get(var_col, "#1f77b4")
    
    st_df = pd.read_csv(data_dir / f"station_{st_id}.csv")
    st_df["datetime"] = pd.to_datetime(st_df["datetime"], format="mixed")
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=st_df["datetime"],
        y=st_df[var_col],
        mode="lines",
        name=var_col,
        line=dict(color=col_color, width=1.4),
    ))
    
    fig.update_layout(
        title=f"{station_selector.label} — {channel_selector.label} Full Time Series",
        xaxis_title="Date / Time",
        yaxis_title=channel_selector.label,
        template="plotly_white",
        hovermode="x",
        height=380,
        margin=dict(l=50, r=20, t=50, b=40),
    )
    fig.show()


interactive(children=(Dropdown(description='Select Station:', layout=Layout(width='260px'), options=(('Station…